### 3.9.22. DFP and BFGS Updates

$$
D^{k+1}
= D^k
+ \left(1+\frac{q^{k\mathsf T}D^kq^k}{p^{k\mathsf T}q^k}\right)
\frac{p^kp^{k\mathsf T}}{p^{k\mathsf T}q^k}
-\frac{D^kq^kp^{k\mathsf T}+p^kq^{k\mathsf T}D^k}{p^{k\mathsf T}q^k}.
$$


**Explanation:**

DFP and BFGS are specific quasi-Newton update formulas for the inverse-Hessian approximation. BFGS is especially important in practice because it tends to be robust and preserves [positive definiteness](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb) when the curvature condition $p^{k\mathsf T}q^k>0$ holds. The update is built from vector outer products, so it incorporates new curvature information while retaining earlier information. Compared with conjugate gradient, it usually needs more storage but is less sensitive to exact line search.

The update constructs the next inverse-Hessian approximation $D^{k+1}$ from the old matrix $D^k$, the step $p^k$, and the gradient change $q^k$. The denominators $p^{k\mathsf T}q^k$ are curvature measurements formed with an [inner product](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/02_inner_product_space.ipynb).

For learners, the formula may look complicated at first, but its purpose is simple. It must satisfy the secant equation, preserve symmetry and positive definiteness, and alter the previous matrix as economically as possible. The update family is best understood through these structural properties rather than as an arbitrary algebraic expression.

**Intuition:**

The figure points to the dual viewpoint: one may update an inverse-Hessian approximation or a Hessian approximation, and the formulas are closely related. The code cell performs one BFGS inverse update and inspects the [eigenvalues](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/08_eigenvalue.ipynb) to see that the resulting matrix is positive definite for the chosen curvature pair.



**Numerical Example:**

The update figure concerns the matrix approximation. Start with
$$
D^k=I,
\qquad p=\begin{bmatrix}1\\0.5\end{bmatrix},
\qquad q=\begin{bmatrix}2\\1.5\end{bmatrix}.
$$

The curvature scalar is
$$
p^{\mathsf T}q=1\cdot 2+0.5\cdot 1.5=2+0.75=2.75>0.
$$

Also,
$$
q^{\mathsf T}D^kq=q^{\mathsf T}q=2^2+1.5^2=4+2.25=6.25.
$$

Substituting these values into the BFGS inverse update gives approximately
$$
D^{k+1}=\begin{bmatrix}0.7355&-0.3140\\-0.3140&0.7521\end{bmatrix}.
$$

The eigenvalues are approximately
$$
0.4296\quad\text{and}\quad 1.0580,
$$
so the updated inverse-Hessian approximation remains positive definite.


In [ ]:
import sympy as sp

inverse_hessian = sp.eye(2)
position_increment = sp.Matrix([1, sp.Rational(1, 2)])
gradient_increment = sp.Matrix([2, sp.Rational(3, 2)])
curvature = (position_increment.T * gradient_increment)[0]
curvature_scaling = 1 + (gradient_increment.T * inverse_hessian * gradient_increment)[0] / curvature
rank_two_correction = (curvature_scaling * (position_increment * position_increment.T)
    - (inverse_hessian * gradient_increment * position_increment.T
       + position_increment * gradient_increment.T * inverse_hessian)) / curvature
bfgs_update = inverse_hessian + rank_two_correction
eigenvalues = sorted(bfgs_update.eigenvals())

print("bfgs_update ="); sp.pprint(bfgs_update)
print("eigenvalues =", eigenvalues)

bfgs_update =
⎡89    -38 ⎤
⎢───   ────⎥
⎢121   121 ⎥
⎢          ⎥
⎢-38   91  ⎥
⎢────  ─── ⎥
⎣121   121 ⎦
eigenvalues = [90/121 - 17*sqrt(5)/121, 17*sqrt(5)/121 + 90/121]


**Equivalent `casadi` implementation:**

In [ ]:
import casadi as ca

inverse_hessian = ca.DM.eye(2)
position_increment = ca.DM([1, 0.5])
gradient_increment = ca.DM([2, 1.5])
curvature = ca.dot(position_increment, gradient_increment)
curvature_scaling = 1 + ca.bilin(inverse_hessian, gradient_increment, gradient_increment) / curvature
rank_two_correction = (curvature_scaling * (position_increment @ position_increment.T)
    - (inverse_hessian @ gradient_increment @ position_increment.T
       + position_increment @ gradient_increment.T @ inverse_hessian)) / curvature
bfgs_update = inverse_hessian + rank_two_correction
eigenvalues = ca.eig_symbolic(ca.SX(bfgs_update))

print("bfgs_update =", bfgs_update)
print("eigenvalues =", eigenvalues)

bfgs_update = 
[[0.735537, -0.31405], 
 [-0.31405, 0.752066]]
eigenvalues = [0.429643, 1.05796]


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.), Chapter 1 "Unconstrained Optimization". Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Quasi-Newton Methods](./21_quasi_newton_methods.ipynb) | [Next: Quasi-Newton Scaling and Comparison ➡️](./23_quasi_newton_scaling_and_comparison.ipynb)

---
